In [2]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

def find_data_directory():
    roots = [Path('/kaggle/input'), Path.cwd(), Path('/mnt/data')]
    for root in roots:
        if not root.is_dir():
            continue
        candidates = [root]
        if root == Path('/kaggle/input'):
            candidates += [p.parent for p in root.rglob('train.csv')]
        for folder in candidates:
            if all((folder / name).is_file() for name in
                   ('train.csv', 'test.csv', 'sample_submission.csv')):
                return folder
    raise FileNotFoundError(
        'Cannot locate train.csv, test.csv, and sample_submission.csv. '
        'In Kaggle click Add Input and add the ML Project 2 competition dataset.'
    )

DATA_DIR = find_data_directory()
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Reading data from:', DATA_DIR)
print('Writing submission to:', OUTPUT_DIR / 'submission.csv')
N_FOLDS = 4
RANDOM_SEED = 42
N_JOBS = 4

# Ensemble weights were selected using out-of-fold economic validation.
RANK_WEIGHT = 0.70
COST_MODEL_WEIGHT = 0.50
COST_SCORE_MULTIPLIER = 2.50
BUDGET_PROB_MULTIPLIER = 1.00
INSPECTION_COST = 10.0
SHORTAGE_RATE = 0.20
SURPLUS_RATE = 0.02

Reading data from: /kaggle/input/competitions/mlprojectfleet2
Writing submission to: /kaggle/working/submission.csv


In [3]:
def load_features():
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    example = pd.read_csv(DATA_DIR / "sample_submission.csv")

    sensors = [c for c in test if c.startswith("s") and c[1:].isdigit()]
    sensors = [c for c in sensors if c in train]

    # A training-only sensor has no signal in the other depot.
    # Remove features that cannot be used in both depots, or are constants.
    sensors = [c for c in sensors
               if not train[c].isna().all()
               and not test[c].isna().all()
               and train[c].nunique(dropna=True) > 1]
    if not sensors:
        raise ValueError("No usable sensor columns were found")

    X_train_raw = train[sensors].astype(np.float32).copy()
    X_test_raw = test[sensors].astype(np.float32).copy()

    # Missingness can itself be predictive. Do not impute; LightGBM handles NaNs.
    X_train_raw["missing_count"] = X_train_raw.isna().sum(axis=1).astype(np.float32)
    X_test_raw["missing_count"] = X_test_raw.isna().sum(axis=1).astype(np.float32)

    # Monotonic-invariant distribution adaptation: rank each sensor separately
    # within its source depot and target depot (using only unlabeled test X).
    X_train_rank = train[sensors].rank(pct=True).astype(np.float32)
    X_test_rank = test[sensors].rank(pct=True).astype(np.float32)
    X_train_rank["missing_count"] = X_train_raw["missing_count"]
    X_test_rank["missing_count"] = X_test_raw["missing_count"]

    labels = train["failure"].to_numpy(dtype=np.int8)
    repair_cost = train["repair_cost"].fillna(0).to_numpy(dtype=float)
    if (repair_cost < 0).any():
        raise ValueError("Negative repair cost encountered")
    if (labels == 1).sum() < N_FOLDS:
        raise ValueError("Not enough observed failures for cross-validation")

    return train, test, example, labels, repair_cost, {
        "raw": (X_train_raw, X_test_raw),
        "rank": (X_train_rank, X_test_rank),
    }

In [4]:
def fit_cv_models(feature_sets, y, repair_cost):
    splits = list(StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED
    ).split(np.zeros(len(y)), y))

    # For estimating cost-weighted failure risk, larger recorded repairs are
    # more important. Clipping dampens the influence of a few extreme costs.
    weights = np.ones(len(y), dtype=float)
    positive = y == 1
    weighted_cost = np.clip(repair_cost[positive], 30.0, 2500.0)
    weights[positive] = weighted_cost / weighted_cost.mean()

    predictions = {}
    for family, (X_train, X_test) in feature_sets.items():
        for cost_weighted in (False, True):
            key = (family, cost_weighted)
            out_of_fold = np.zeros(len(y), dtype=float)
            test_probs = np.zeros(len(X_test), dtype=float)
            rounds = []

            for fold, (train_idx, val_idx) in enumerate(splits, 1):
                model = lgb.LGBMClassifier(
                    objective="binary",
                    n_estimators=700,
                    learning_rate=0.035,
                    num_leaves=15,
                    min_child_samples=65,
                    reg_lambda=8.0,
                    colsample_bytree=0.85,
                    subsample=0.85,
                    subsample_freq=1,
                    random_state=RANDOM_SEED + fold,
                    n_jobs=N_JOBS,
                    verbosity=-1,
                )

                fit_args = {
                    "eval_set": [(X_train.iloc[val_idx], y[val_idx])],
                    "eval_metric": "binary_logloss",
                    "callbacks": [
                        lgb.early_stopping(70, first_metric_only=True, verbose=False)
                    ],
                }
                if cost_weighted:
                    fit_args["sample_weight"] = weights[train_idx]

                model.fit(X_train.iloc[train_idx], y[train_idx], **fit_args)
                out_of_fold[val_idx] = model.predict_proba(
                    X_train.iloc[val_idx]
                )[:, 1]
                test_probs += model.predict_proba(X_test)[:, 1] / N_FOLDS
                rounds.append(model.best_iteration_)

            predictions[key] = (out_of_fold, test_probs)
            mode = "cost-weighted" if cost_weighted else "standard"
            print(f"{family:4} {mode:13} | "
                  f"ROC-AUC={roc_auc_score(y, out_of_fold):.5f} "
                  f"PR-AUC={average_precision_score(y, out_of_fold):.5f} "
                  f"rounds={rounds}")
    return predictions

In [5]:
def ensemble_predictions(predictions, use_test):
    side = 1 if use_test else 0
    standard = (
        (1.0 - RANK_WEIGHT) * predictions[("raw", False)][side]
        + RANK_WEIGHT * predictions[("rank", False)][side]
    )
    cost_weighted = (
        (1.0 - RANK_WEIGHT) * predictions[("raw", True)][side]
        + RANK_WEIGHT * predictions[("rank", True)][side]
    )
    # A weighted classifier is not a calibrated true-failure probability.
    # Keep the standard estimator for budgeting, and use the blended score
    # to prioritize potentially high-cost failures for inspection.
    cost_score = (
        (1.0 - COST_MODEL_WEIGHT) * standard
        + COST_MODEL_WEIGHT * cost_weighted
    )
    return standard, cost_score


def choose_inspections(probability, cost_score, observed_positive_costs):
    """Analytic spare-parts budget and expected-cost inspection decisions.

    Under a constant conditional repair-cost distribution, the optimal budget
    is the 0.20/(0.20+0.02) = 90.91% quantile of the *unconditional* parts
    requirement distribution, which includes a point mass at zero for
    non-failures. This gives the conditional quantile level
       1 - (0.02 / 0.22) / P(failure).
    """
    p = np.clip(BUDGET_PROB_MULTIPLIER * probability, 0.0, 1.0)
    risk = np.clip(COST_SCORE_MULTIPLIER * cost_score, 0.0, 1.0)

    positive_costs = np.sort(np.asarray(observed_positive_costs, dtype=float))
    count = positive_costs.size
    if count == 0:
        raise ValueError("No positive repairs found")
    cumulative_cost = np.concatenate(([0.0], np.cumsum(positive_costs)))

    quantile = 1.0 - SURPLUS_RATE / (SHORTAGE_RATE + SURPLUS_RATE) / np.maximum(p, 1e-12)
    budget = np.zeros(len(p), dtype=float)
    needs_parts = quantile > 0
    selected_idx = np.ceil(quantile[needs_parts] * count).astype(int) - 1
    budget[needs_parts] = positive_costs[np.clip(selected_idx, 0, count - 1)]

    # Estimate conditional expected shortage and conditional expected surplus
    # using empirical observed repair costs; non-failures have zero demand.
    k = np.searchsorted(positive_costs, budget, side="right")
    expected_shortage = (cumulative_cost[-1] - cumulative_cost[k] -
                         (count - k) * budget) / count
    expected_positive_surplus = (k * budget - cumulative_cost[k]) / count

    expected_inspection_cost = (
        INSPECTION_COST
        + SHORTAGE_RATE * p * expected_shortage
        + SURPLUS_RATE * ((1.0 - p) * budget + p * expected_positive_surplus)
    )

    # Using cost-weighted ranking, estimate the cost of skipping inspection.
    expected_skipping_cost = risk * positive_costs.mean()
    flag = expected_inspection_cost < expected_skipping_cost

    final_budget = np.where(flag, budget, 0.0)
    return flag.astype(np.int64), np.rint(final_budget).astype(np.int64)


def estimated_cost(flag, budget, true_cost):
    """Internal validation proxy based on the visible competition cost terms."""
    return float(np.sum(
        INSPECTION_COST * flag
        + flag * (SHORTAGE_RATE * np.maximum(true_cost - budget, 0.0)
                  + SURPLUS_RATE * np.maximum(budget - true_cost, 0.0))
        + (1 - flag) * true_cost
    ))

In [6]:
def main():
    train, test, example, y, repair_cost, feature_sets = load_features()
    print(f"Training rows: {len(train)}, test rows: {len(test)}, "
          f"failures: {y.sum()} ({y.mean():.3%})")

    predictions = fit_cv_models(feature_sets, y, repair_cost)
    positive_costs = repair_cost[y == 1]

    oof_prob, oof_score = ensemble_predictions(predictions, use_test=False)
    valid_flag = np.zeros(len(y), dtype=np.int64)
    valid_budget = np.zeros(len(y), dtype=np.int64)
  
    oof_splits = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED
    )
    for fit_idx, valid_idx in oof_splits.split(np.zeros(len(y)), y):
        fold_positive_costs = repair_cost[fit_idx][y[fit_idx] == 1]
        fold_flag, fold_budget = choose_inspections(
            oof_prob[valid_idx], oof_score[valid_idx], fold_positive_costs
        )
        valid_flag[valid_idx] = fold_flag
        valid_budget[valid_idx] = fold_budget
    print("\nOut-of-fold validation proxy (lower is better):")
    print(f"  Never inspect:      {repair_cost.sum():,.2f}")
    print(f"  Cost-based strategy: {estimated_cost(valid_flag, valid_budget, repair_cost):,.2f}")
    print(f"  Vehicles inspected: {int(valid_flag.sum()):,} / {len(train):,}")

    test_prob, test_score = ensemble_predictions(predictions, use_test=True)
    flag, budget = choose_inspections(test_prob, test_score, positive_costs)

    # Preserve the exact sample-submission row order and required column names.
    prediction_frame = pd.DataFrame({
        "id": test["id"], "flag": flag, "budget": budget
    })
    assert prediction_frame["id"].is_unique
    assert example["id"].is_unique
    submission = example[["id"]].merge(
        prediction_frame, on="id", how="left", validate="one_to_one"
    )
    assert submission["flag"].notna().all()
    assert len(submission) == len(test) == len(example)
    assert (submission["budget"] >= 0).all()
    assert set(submission["flag"].unique()).issubset({0, 1})
    assert (submission.loc[submission["flag"] == 0, "budget"] == 0).all()
    path = OUTPUT_DIR / "submission.csv"
    submission[["id", "flag", "budget"]].to_csv(path, index=False)

    print(f"\nSubmission saved: {path}")
    print(f"  Rows: {len(submission):,}")
    print(f"  Inspections: {submission['flag'].sum():,}")
    print(f"  Nonzero spare-parts budgets: {(submission['budget'] > 0).sum():,}")
    print(f"  Total allocated parts budget: {submission['budget'].sum():,.0f}")
    print(submission.head().to_string(index=False))

In [7]:
main()

Training rows: 56000, test rows: 20000, failures: 397 (0.709%)
raw  standard      | ROC-AUC=0.98791 PR-AUC=0.52350 rounds=[262, 224, 124, 183]
raw  cost-weighted | ROC-AUC=0.98655 PR-AUC=0.50489 rounds=[560, 387, 137, 164]
rank standard      | ROC-AUC=0.98739 PR-AUC=0.52731 rounds=[262, 176, 134, 183]
rank cost-weighted | ROC-AUC=0.98648 PR-AUC=0.50066 rounds=[500, 424, 137, 179]

Out-of-fold validation proxy (lower is better):
  Never inspect:      195,476.00
  Cost-based strategy: 57,594.22
  Vehicles inspected: 2,056 / 56,000

Submission saved: /kaggle/working/submission.csv
  Rows: 20,000
  Inspections: 1,164
  Nonzero spare-parts budgets: 622
  Total allocated parts budget: 256,257
    id  flag  budget
t00001     0       0
t00002     0       0
t00003     0       0
t00004     0       0
t00005     0       0
